<a href="https://colab.research.google.com/github/Seeratfaatima/AWS-MLOps-Production-Pipeline/blob/main/AI_Powered_Semantic_FAQ_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1 — Install libraries

In [1]:
!pip install -q gensim sentence-transformers scikit-learn pandas numpy

# Cell 2 — Import libraries

In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec, FastText
from sentence_transformers import SentenceTransformer

# Cell 3 — Create the FAQ knowledge base

In [3]:
faqs = [
    {
        "question": "How can I apply for university admission?",
        "answer": "Apply through the university's online admission portal. Complete the form, upload required documents, and submit the application fee.",
        "category": "Admissions"
    },
    {
        "question": "What documents are required for admission?",
        "answer": "Common documents include educational transcripts, degree or certificate, CNIC or B-Form, photograph, and application form.",
        "category": "Admissions"
    },
    {
        "question": "What is the last date to apply?",
        "answer": "The deadline depends on the admission session. Check the official admission portal for the latest date.",
        "category": "Admissions"
    },
    {
        "question": "How can I check my admission application status?",
        "answer": "Log in to the admission portal and check the application status section.",
        "category": "Admissions"
    },
    {
        "question": "What are the eligibility criteria for computer science?",
        "answer": "Eligibility depends on your previous qualification, required subjects, minimum marks, and university admission policy.",
        "category": "Programs"
    },
    {
        "question": "Does the university offer a bachelor's degree in artificial intelligence?",
        "answer": "Check the university's undergraduate programs page to confirm whether a bachelor's degree in artificial intelligence is offered.",
        "category": "Programs"
    },
    {
        "question": "How much is the semester fee?",
        "answer": "Semester fees vary by degree program and academic session. Check the official fee structure for the current amount.",
        "category": "Fees"
    },
    {
        "question": "Are scholarships available for students?",
        "answer": "Scholarships may be available based on merit, financial need, or specific eligibility criteria. Contact the financial aid office for details.",
        "category": "Scholarships"
    },
    {
        "question": "How can I apply for financial assistance?",
        "answer": "Contact the financial aid office and submit the financial assistance application with supporting documents.",
        "category": "Scholarships"
    },
    {
        "question": "Where can I find the university campus location?",
        "answer": "The campus address and directions are available on the university's official website.",
        "category": "Campus"
    },
    {
        "question": "Does the university provide hostel facilities?",
        "answer": "Hostel availability, eligibility, and fees depend on the university. Contact the student affairs or hostel office.",
        "category": "Campus"
    },
    {
        "question": "How can I contact the university?",
        "answer": "Official phone numbers, email addresses, and contact forms are available on the university's website.",
        "category": "Contact"
    }
]

df = pd.DataFrame(faqs)
df

,question,answer,category
0,How can I apply for university admission?,Apply through the university's online admissio...,Admissions
1,What documents are required for admission?,Common documents include educational transcrip...,Admissions
2,What is the last date to apply?,The deadline depends on the admission session....,Admissions
3,How can I check my admission application status?,Log in to the admission portal and check the a...,Admissions
4,What are the eligibility criteria for computer...,Eligibility depends on your previous qualifica...,Programs
5,Does the university offer a bachelor's degree ...,Check the university's undergraduate programs ...,Programs
6,How much is the semester fee?,Semester fees vary by degree program and acade...,Fees
7,Are scholarships available for students?,"Scholarships may be available based on merit, ...",Scholarships
8,How can I apply for financial assistance?,Contact the financial aid office and submit th...,Scholarships
9,Where can I find the university campus location?,The campus address and directions are availabl...,Campus


# Cell 4 — Preprocessing and tokenization

In [8]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return preprocess_text(text).split()

df["clean_question"] = df["question"].apply(preprocess_text)
df["tokens"] = df["clean_question"].apply(tokenize)

df[["question", "clean_question", "tokens"]].head(12)

,question,clean_question,tokens
0,How can I apply for university admission?,how can i apply for university admission,"[how, can, i, apply, for, university, admission]"
1,What documents are required for admission?,what documents are required for admission,"[what, documents, are, required, for, admission]"
2,What is the last date to apply?,what is the last date to apply,"[what, is, the, last, date, to, apply]"
3,How can I check my admission application status?,how can i check my admission application status,"[how, can, i, check, my, admission, applicatio..."
4,What are the eligibility criteria for computer...,what are the eligibility criteria for computer...,"[what, are, the, eligibility, criteria, for, c..."
5,Does the university offer a bachelor's degree ...,does the university offer a bachelor s degree ...,"[does, the, university, offer, a, bachelor, s,..."
6,How much is the semester fee?,how much is the semester fee,"[how, much, is, the, semester, fee]"
7,Are scholarships available for students?,are scholarships available for students,"[are, scholarships, available, for, students]"
8,How can I apply for financial assistance?,how can i apply for financial assistance,"[how, can, i, apply, for, financial, assistance]"
9,Where can I find the university campus location?,where can i find the university campus location,"[where, can, i, find, the, university, campus,..."


# Cell 5 — One-hot encoding demonstration

In [5]:
sample_words = np.array([
    ["admission"],
    ["university"],
    ["scholarship"]
])

# Compatibility with different scikit-learn versions
try:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

one_hot_matrix = encoder.fit_transform(sample_words)

print("Words:")
print(sample_words.ravel())

print("\nOne-hot vectors:")
print(one_hot_matrix)

print("\nVocabulary:")
print(encoder.categories_[0])

Words:
['admission' 'university' 'scholarship']

One-hot vectors:
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]

Vocabulary:
['admission' 'scholarship' 'university']


# Cell 6 — Bag of Words

In [7]:
bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(df["clean_question"])

print("BoW matrix shape:", bow_matrix.shape)
print("Vocabulary size:", len(bow_vectorizer.get_feature_names_out()))

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

bow_df.head(12)

BoW matrix shape: (12, 46)
Vocabulary size: 46


,admission,application,apply,are,artificial,assistance,available,bachelor,campus,can,...,scholarships,science,semester,status,students,the,to,university,what,where
0,1,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
1,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,1,0
3,1,1,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,1,0
5,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,1,0,1,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,0,0
7,0,0,0,1,0,0,1,0,0,0,...,1,0,0,0,1,0,0,0,0,0
8,0,0,1,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,1,0,1,0,1


# Cell 7 — TF-IDF

In [10]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(df["clean_question"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_df.head(12)

TF-IDF matrix shape: (12, 46)


,admission,application,apply,are,artificial,assistance,available,bachelor,campus,can,...,scholarships,science,semester,status,students,the,to,university,what,where
0,0.463755,0.000000,0.463755,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.377446,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.377446,0.000000,0.000000
1,0.374307,0.000000,0.000000,0.374307,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.374307,0.000000
2,0.000000,0.000000,0.334095,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.227801,0.440388,0.000000,0.334095,0.000000
3,0.328355,0.432823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.267246,...,0.000000,0.000000,0.000000,0.432823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.315010,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.415232,0.000000,0.000000,0.000000,0.214789,0.000000,0.000000,0.315010,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.367946,0.000000,0.000000,0.367946,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.190329,0.000000,0.227188,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.477471,0.000000,0.000000,0.246983,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,0.381385,0.000000,0.000000,0.502724,0.000000,0.000000,0.000000,...,0.502724,0.000000,0.000000,0.000000,0.502724,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.393374,0.000000,0.000000,0.518528,0.000000,0.000000,0.000000,0.320164,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.445875,0.275305,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.230640,0.000000,0.275305,0.000000,0.445875


# Cell 8 — Train Word2Vec

In [13]:
tokenized_questions = df["tokens"].tolist()

word2vec_model = Word2Vec(
    sentences=tokenized_questions,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,       # 1 = Skip-gram; 0 = CBOW
    epochs=100
)

word = "admission"

print("Vector for:", word)
print(word2vec_model.wv[word][:10])

print("\nSimilar words:")
print(word2vec_model.wv.most_similar(word, topn=5))

Vector for: admission
[-0.01490721  0.00457151 -0.01025871  0.00026075  0.00648036  0.00871561
  0.01004784  0.01024993 -0.01190035  0.00968673]

Similar words:
[('s', 0.4330052435398102), ('for', 0.3862100839614868), ('does', 0.35389435291290283), ('offer', 0.3422509431838989), ('semester', 0.3411577045917511)]


# Cell 9 — Train FastText


In [14]:
fasttext_model = FastText(
    sentences=tokenized_questions,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    epochs=100
)

print("FastText vector for 'admission':")
print(fasttext_model.wv["admission"][:10])

# FastText can generate a vector using character n-grams
# even if the exact word was not in the training vocabulary.
print("\nVector for 'admissions':")
print(fasttext_model.wv["admissions"][:10])

FastText vector for 'admission':
[-0.00141329 -0.00250291  0.00759954  0.00040395 -0.00301987  0.00451633
 -0.00186704  0.00918334 -0.00255909 -0.00038497]

Vector for 'admissions':
[-1.8239974e-03 -1.1869993e-03  6.1171032e-03  6.4408599e-04
 -2.9331241e-03  4.8192050e-03 -2.9621301e-03  4.4206027e-03
 -3.3903178e-03  4.1043535e-05]


# Cell 10 — Generate sentence embeddings

In [15]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

questions = df["question"].tolist()

sentence_embeddings = sentence_model.encode(
    questions,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:", sentence_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix shape: (12, 384)


# Cell 11 — Create search functions

In [16]:
def search_bow(user_query, top_k=3):
    query = preprocess_text(user_query)
    query_vector = bow_vectorizer.transform([query])

    scores = cosine_similarity(query_vector, bow_matrix)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return ranked_indices, scores


def search_tfidf(user_query, top_k=3):
    query = preprocess_text(user_query)
    query_vector = tfidf_vectorizer.transform([query])

    scores = cosine_similarity(query_vector, tfidf_matrix)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return ranked_indices, scores


def search_semantic(user_query, top_k=3):
    query_embedding = sentence_model.encode(
        [user_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Normalized embeddings: dot product gives cosine similarity
    scores = np.dot(sentence_embeddings, query_embedding[0])

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return ranked_indices, scores

# Cell 12 — Display matching FAQs and answer

In [17]:
def display_results(user_query, method="semantic", top_k=3, threshold=0.35):

    if method == "bow":
        ranked_indices, scores = search_bow(user_query, top_k)

    elif method == "tfidf":
        ranked_indices, scores = search_tfidf(user_query, top_k)

    else:
        ranked_indices, scores = search_semantic(user_query, top_k)

    print("\nUser question:", user_query)
    print("Search method:", method.upper())
    print("=" * 60)

    for rank, index in enumerate(ranked_indices, start=1):
        print(f"\nRank {rank}")
        print("FAQ:", df.iloc[index]["question"])
        print("Category:", df.iloc[index]["category"])
        print("Similarity:", round(float(scores[index]), 4))

    best_index = ranked_indices[0]
    best_score = scores[best_index]

    print("\n" + "=" * 60)
    print("BEST MATCH")

    if best_score < threshold:
        print("No sufficiently relevant FAQ found.")
        print("Please contact the university for assistance.")
    else:
        print("Question:", df.iloc[best_index]["question"])
        print("Answer:", df.iloc[best_index]["answer"])
        print("Similarity:", round(float(best_score), 4))

# Cell 13 — Test one question

In [18]:
user_query = "What papers do I need to submit to get into university?"

display_results(
    user_query,
    method="semantic",
    top_k=3
)


User question: What papers do I need to submit to get into university?
Search method: SEMANTIC

Rank 1
FAQ: What documents are required for admission?
Category: Admissions
Similarity: 0.7645

Rank 2
FAQ: How can I apply for university admission?
Category: Admissions
Similarity: 0.618

Rank 3
FAQ: How can I contact the university?
Category: Contact
Similarity: 0.4465

BEST MATCH
Question: What documents are required for admission?
Answer: Common documents include educational transcripts, degree or certificate, CNIC or B-Form, photograph, and application form.
Similarity: 0.7645


In [19]:
user_query = "What papers do I need to submit to get into university?"

display_results(
    user_query,
    method="bow",
    top_k=3
)


User question: What papers do I need to submit to get into university?
Search method: BOW

Rank 1
FAQ: What is the last date to apply?
Category: Admissions
Similarity: 0.4629

Rank 2
FAQ: How can I contact the university?
Category: Contact
Similarity: 0.1826

Rank 3
FAQ: What documents are required for admission?
Category: Admissions
Similarity: 0.1667

BEST MATCH
Question: What is the last date to apply?
Answer: The deadline depends on the admission session. Check the official admission portal for the latest date.
Similarity: 0.4629


In [20]:
user_query = "What papers do I need to submit to get into university?"

display_results(
    user_query,
    method="tfidf",
    top_k=3
)


User question: What papers do I need to submit to get into university?
Search method: TFIDF

Rank 1
FAQ: What is the last date to apply?
Category: Admissions
Similarity: 0.5095

Rank 2
FAQ: What documents are required for admission?
Category: Admissions
Similarity: 0.1275

Rank 3
FAQ: How can I contact the university?
Category: Contact
Similarity: 0.1103

BEST MATCH
Question: What is the last date to apply?
Answer: The deadline depends on the admission session. Check the official admission portal for the latest date.
Similarity: 0.5095


# Cell 14 — Interactive question-answer loop

In [21]:
while True:
    user_query = input("\nAsk a university question (type 'exit' to stop): ")

    if user_query.lower().strip() == "exit":
        print("Goodbye!")
        break

    method = input("Choose method (semantic/tfidf/bow): ").lower().strip()

    if method not in ["semantic", "tfidf", "bow"]:
        method = "semantic"

    display_results(
        user_query,
        method=method,
        top_k=3
    )


Ask a university question (type 'exit' to stop): Which university should I apply?
Choose method (semantic/tfidf/bow): semantic

User question: Which university should I apply?
Search method: SEMANTIC

Rank 1
FAQ: How can I apply for university admission?
Category: Admissions
Similarity: 0.6215

Rank 2
FAQ: How can I contact the university?
Category: Contact
Similarity: 0.5107

Rank 3
FAQ: Where can I find the university campus location?
Category: Campus
Similarity: 0.4498

BEST MATCH
Question: How can I apply for university admission?
Answer: Apply through the university's online admission portal. Complete the form, upload required documents, and submit the application fee.
Similarity: 0.6215

Ask a university question (type 'exit' to stop): How to apply?
Choose method (semantic/tfidf/bow): semantic

User question: How to apply?
Search method: SEMANTIC

Rank 1
FAQ: How can I apply for university admission?
Category: Admissions
Similarity: 0.6317

Rank 2
FAQ: What is the last date to a

# Cell 15 — Evaluate on test queries

In [22]:
test_queries = [
    ("How do I get admission?", 0),
    ("Which documents should I submit?", 1),
    ("When does the application close?", 2),
    ("How do I see my application progress?", 3),
    ("What grades do I need for computer science?", 4),
    ("Can I study artificial intelligence?", 5),
    ("What is the cost per semester?", 6),
    ("Can students receive scholarships?", 7),
    ("I need help paying my tuition.", 8),
    ("Where is the campus situated?", 9),
    ("Can I stay in a student hostel?", 10),
    ("What is the university's email or phone number?", 11)
]

correct = 0

for query, expected_index in test_queries:
    ranked_indices, scores = search_semantic(query, top_k=1)
    predicted_index = ranked_indices[0]

    is_correct = predicted_index == expected_index
    correct += int(is_correct)

    print("\nQuery:", query)
    print("Predicted FAQ:", df.iloc[predicted_index]["question"])
    print("Correct:", is_correct)

accuracy = correct / len(test_queries)

print("\nTop-1 Accuracy:", round(accuracy * 100, 2), "%")


Query: How do I get admission?
Predicted FAQ: How can I apply for university admission?
Correct: True

Query: Which documents should I submit?
Predicted FAQ: What documents are required for admission?
Correct: True

Query: When does the application close?
Predicted FAQ: What is the last date to apply?
Correct: True

Query: How do I see my application progress?
Predicted FAQ: How can I check my admission application status?
Correct: True

Query: What grades do I need for computer science?
Predicted FAQ: What are the eligibility criteria for computer science?
Correct: True

Query: Can I study artificial intelligence?
Predicted FAQ: Does the university offer a bachelor's degree in artificial intelligence?
Correct: True

Query: What is the cost per semester?
Predicted FAQ: How much is the semester fee?
Correct: True

Query: Can students receive scholarships?
Predicted FAQ: Are scholarships available for students?
Correct: True

Query: I need help paying my tuition.
Predicted FAQ: How much